In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

pd.set_option("display.width", 140)
%matplotlib inline

## Data

In [31]:
a_df = pd.read_csv("gbpusd-m15-bid-2020-01-01-2026-08-18.csv")
b_df = pd.read_csv("eurusd-m15-bid-2020-01-01-2026-08-18.csv")

a_df.set_index("Datetime", inplace=True)
b_df.set_index("Datetime", inplace=True)

a_df = a_df[["Open", "High", "Low", "Close"]]
b_df = b_df[["Open", "High", "Low", "Close"]]

a_df.index = pd.to_datetime(a_df.index, unit="ms")
b_df.index = pd.to_datetime(b_df.index, unit="ms")

df = pd.concat(
    [
        a_df["Close"].rename("A"),
        b_df["Close"].rename("B")
    ],
    axis=1,
    join="inner"
)

df = df[df.index < '2022-01-01']

df

,A,B
Datetime,,
2020-01-01 00:00:00,1.32463,1.12139
2020-01-01 00:15:00,1.32463,1.12139
2020-01-01 00:30:00,1.32463,1.12139
2020-01-01 00:45:00,1.32463,1.12139
2020-01-01 01:00:00,1.32463,1.12139
...,...,...
2021-12-31 20:45:00,1.35233,1.13760
2021-12-31 21:00:00,1.35308,1.13786
2021-12-31 21:15:00,1.35217,1.13786


In [32]:
from statsmodels.regression.rolling import RollingOLS

# -------------- Model --------------
def pair_spread_mean_reversion_model(window):
    Y = df["A"]

    X = sm.add_constant(df["B"])
    rolling_model = RollingOLS(Y, X, window=window)
    fit = rolling_model.fit()

    _beta = fit.params["B"]
    _alpha = fit.params["const"]
    _spread = (df["A"] - (_alpha + _beta * df["B"])).dropna()
    _beta = _beta.reindex(_spread.index)

    return _alpha, _beta, _spread

# stability of β

In [33]:
n_chunks = [48, 100] # check beta-stability.py

beta_stability_test_results = {}

for n in n_chunks:
    beta_stability_test_results[n] = []

    for chunk in (df.iloc[i:i+n] for i in range(0, len(df), n)):
        if len(chunk) < n: continue

        col_a = "A"
        col_b = "B"

        Y = chunk[col_a]
        X = sm.add_constant(chunk[col_b])

        fit = sm.OLS(Y,X).fit()

        beta = fit.params[col_b]

        beta_stability_test_results[n].append(beta)

In [34]:
for n, betas in beta_stability_test_results.items():

    betas = np.asarray(betas)

    mean_beta = np.mean(betas)
    std_beta = np.std(betas, ddof=1)

    # Relative variability
    cv = std_beta / abs(mean_beta)

    # Total spread of beta
    beta_range = np.max(betas) - np.min(betas)

    # Average change from one period to the next
    avg_drift = np.mean(np.abs(np.diff(betas)))

    beta_stability_test_results[n] = {
        "mean_beta": mean_beta,
        "std_beta": std_beta,
        "cv_pct": cv * 100,
        "min_beta": np.min(betas),
        "max_beta": np.max(betas),
        "range": beta_range,
        "avg_drift": avg_drift,
        "n_periods": len(betas),
    }

beta_stability_test_results_df = pd.DataFrame(beta_stability_test_results).T

beta_stability_test_results_df

,mean_beta,std_beta,cv_pct,min_beta,max_beta,range,avg_drift,n_periods
48,0.939114,0.945068,100.633949,-8.059623,6.391479,14.451102,0.944516,1227.0
100,0.914514,0.909687,99.472232,-2.843871,5.192007,8.035878,0.942464,589.0


- read data from test (check beta-stability.py run on VPS)

In [35]:
beta_stability_test_results_df = pd.read_csv("tests/stability.csv")

beta_stability_test_results_df.set_index("n_chunks", inplace=True)

beta_stability_test_results_df

,mean_beta,std_beta,cv_pct,min_beta,max_beta,range,avg_drift,n_periods
n_chunks,,,,,,,,
20,0.951457,0.827117,86.931629,-5.013993,7.065689,12.079682,0.818247,2946.0
21,0.936507,1.006934,107.520278,-24.000000,7.900000,31.900000,0.869840,2805.0
22,0.962752,0.835164,86.747578,-3.635643,6.132803,9.768447,0.833002,2678.0
23,0.938759,1.070719,114.056958,-22.125000,6.044805,28.169805,0.883544,2561.0
24,0.957962,0.894950,93.422322,-7.664565,11.049188,18.713754,0.879123,2455.0
...,...,...,...,...,...,...,...,...
995,0.881271,0.711680,80.756064,-0.869529,2.185018,3.054547,0.738904,59.0
996,0.889599,0.724441,81.434592,-0.851076,2.366377,3.217452,0.758226,59.0
997,0.899361,0.741480,82.445139,-0.855532,2.590799,3.446331,0.776263,59.0


- Lower CV = more stable β.

In [36]:
fig = go.Figure(
    go.Bar(
        x=beta_stability_test_results_df.index,
        y=beta_stability_test_results_df["cv_pct"],
        name="Beta CV"
    )
)

fig.update_layout(
    title="Beta Stability vs Chunk Size",
    xaxis_title="Chunk Size (bars)",
    yaxis_title="Beta CV (%)",
    template="plotly_dark",
    width=1500,
    height=500
)

fig.show()

- Best model windows

In [37]:
beta_stability_test_results_top_10 = beta_stability_test_results_df.nsmallest(10, "cv_pct")
[round(len(df)/i) for i in list(beta_stability_test_results_top_10.index)]

[71, 72, 63, 63, 68, 69, 68, 68, 63, 71]

# Half life

|     Half-life |           Time | My interpretation                    |
| ------------: | -------------: | ------------------------------------ |
|      1–2 bars |      15–30 min | ⚠️ Probably too fast/noisy           |
|  **3–8 bars** | **45 min–2 h** | ⭐ Very interesting                   |
| **8–16 bars** |      **2–4 h** | ⭐ Good                               |
|    16–32 bars |          4–8 h | Acceptable, slower                   |
|    32–64 bars |         8–16 h | ⚠️ Probably too slow for intraday MR |
|      >64 bars |          >16 h | ❌ I'd generally avoid                |


In [38]:
sample_windows = [71] # check tests ran on VPS

def half_life(spread):
    spread = spread.dropna()

    lagged = spread.shift(1)
    delta = spread - lagged

    data = sm.add_constant(lagged)

    model = sm.OLS(delta, data, missing="drop").fit()

    lambda_ = model.params.iloc[1]

    if lambda_ >= 0: return np.inf

    return -np.log(2) / lambda_

hal_life_test_results = {}

for window in sample_windows:
    # Example
    alpha, beta, spread = pair_spread_mean_reversion_model(window)

    hl = half_life(spread)

    hal_life_test_results[window] = hl

    print(f"For {window} - Half-life: {hl:.2f} periods ie: {hl*4:.2f} hours")

For 71 - Half-life: 12.12 periods ie: 48.47 hours


In [39]:
half_life_results_df = pd.read_csv("tests/half_life_results.csv")
half_life_results_df.set_index("window", inplace=True)
half_life_results_df

,half_life
window,
60,19.065148
65,15.930030
70,11.144039
75,9.213254
80,6.945517
...,...
980,187.091601
985,188.143736
990,189.300022


In [40]:
fig = go.Figure(
    go.Bar(
        x=half_life_results_df.index,
        y=half_life_results_df["half_life"],
        name="Half life"
    )
)

fig.update_layout(
    title="Half life of window",
    xaxis_title="window",
    yaxis_title="Half life (in Bars)",
    template="plotly_dark",
    width=1500,
    height=500
)

fig.show()

- Best model windows

In [41]:
half_life_results_top_10 = half_life_results_df.nsmallest(10, "half_life")
list(half_life_results_top_10.index)

[85, 80, 75, 70, 65, 90, 95, 60, 100, 105]

# stationarity of the spread ADF test

In [42]:
from statsmodels.tsa.stattools import adfuller

def ADF_test(_spread):
    return adfuller(_spread, autolag="AIC")

sample_windows = [71] # check tests ran on VPS

adf_test_results = {}

for mw in sample_windows:
    _alpha, _beta, _spread = pair_spread_mean_reversion_model(mw)

    _test_result = ADF_test(_spread.dropna())

    stat, pvalue, lags, nobs, crit_values, _ = _test_result

    _is_valid = pvalue < 0.05

    if not _is_valid: continue

    adf_test_results[mw] = {
        "stat":stat ,
        "p-value":pvalue 
    }

    log_phrase =  f"For windows: {mw}, ADF test: statistic={stat:.4f}, p-value={pvalue:.4g}, lags={lags}, nobs={nobs}"

    print(log_phrase)

For windows: 71, ADF test: statistic=-27.4154, p-value=0, lags=60, nobs=58793


In [43]:
adf_test_results_df = pd.read_csv("tests/adf-test.csv")
adf_test_results_df.set_index("window", inplace=True)
adf_test_results_df

,stat,p-value
window,,
50,-38.935862,0.0
51,-38.328924,0.0
52,-35.070073,0.0
53,-33.557327,0.0
54,-34.619990,0.0
...,...,...
146,-33.431137,0.0
147,-33.363458,0.0
148,-33.290526,0.0


In [44]:
adf_test_results_top_10 = adf_test_results_df.sort_values(
    "stat"
).head(10)

list(adf_test_results_top_10.index)

[82, 84, 79, 91, 89, 80, 87, 88, 90, 83]

# Combine all results

In [45]:
results = [round(len(df)/i) for i in list(beta_stability_test_results_top_10.index)] + list(adf_test_results_top_10.index) + list(half_life_results_top_10.index)

[min(results), max(results)]

[60, 105]

# Performance test

In [2]:
performance_test_results_df = pd.read_csv("tests/performance-test.csv").dropna()
performance_test_results_df.set_index(["mw", "zw", "ze", "zf", "zs"], inplace=True)
performance_test_results_df.sort_values("Return %", inplace=True, ascending=False)
performance_test_results_df.head(10)

Return %  Max Drawdown %  Sharpe ratio  Win Rate %  Number of trades   Max win  Max loss  p-value      stat
mw zw ze  zf  zs                                                                                                              
60 41 4.0 0.0 4.7   67.8022         -6.2637          0.68       50.48             44459  0.040296 -0.009443      0.0 -25.04355
   38 3.9 0.0 4.6   66.9662         -7.7217          0.67       50.36             44627  0.039811 -0.009396      0.0 -25.04355
   39 3.9 0.0 4.6   65.8830         -6.2637          0.66       50.34             44921  0.040193 -0.009335      0.0 -25.04355
   37 3.9 0.0 4.6   65.0057         -7.7217          0.65       50.40             44764  0.039152 -0.009286      0.0 -25.04355
   38 3.9 0.0 4.7   64.8316         -7.7217          0.65       50.40             45735  0.038522 -0.009276      0.0 -25.04355
   39 3.1 0.0 3.8   64.8094         -8.6490          0.65       50.41             44827  0.040248 -0.009152      0.0 -25.04355
              3.9   64.3621         -6.4738          0.64       50.40             46750  0.040690 -0.009355      0.0 -25.04355
   41 3.1 0.0 3.9   64.3229         -8.8022          0.64       50.43             46227  0.039716 -0.009067      0.0 -25.04355
   39 3.9 0.0 4.8   63.8426         -6.2637          0.64       50.35             45857  0.039301 -0.009335      0.0 -25.04355
   42 4.0 0.0 4.7   63.7446         -6.2637          0.64       50.40             44938  0.039592 -0.009260      0.0 -25.04355

In [ ]:
best20 = performance_test_results_df.sort_values("Sharpe ratio", ascending=False).head(20)

best20.to_html("tests/15min-best-setups.html")
best20 = best20.reset_index()
best20

,mw,zw,ze,zf,zs,Return %,Max Drawdown %,Sharpe ratio,Win Rate %,Number of trades,Max win,Max loss,p-value,stat
0,60,41,4.0,0.0,4.7,67.8022,-6.2637,0.68,50.48,44459,0.040296,-0.009443,0.0,-25.04355
1,60,38,3.9,0.0,4.6,66.9662,-7.7217,0.67,50.36,44627,0.039811,-0.009396,0.0,-25.04355
2,60,39,3.9,0.0,4.6,65.8830,-6.2637,0.66,50.34,44921,0.040193,-0.009335,0.0,-25.04355
3,60,38,3.9,0.0,4.7,64.8316,-7.7217,0.65,50.40,45735,0.038522,-0.009276,0.0,-25.04355
4,60,39,3.1,0.0,3.8,64.8094,-8.6490,0.65,50.41,44827,0.040248,-0.009152,0.0,-25.04355
5,60,37,3.9,0.0,4.6,65.0057,-7.7217,0.65,50.40,44764,0.039152,-0.009286,0.0,-25.04355
6,60,39,3.9,0.0,4.8,63.8426,-6.2637,0.64,50.35,45857,0.039301,-0.009335,0.0,-25.04355
7,60,42,4.0,0.0,4.7,63.7446,-6.2637,0.64,50.40,44938,0.039592,-0.009260,0.0,-25.04355
8,60,39,3.1,0.0,3.9,64.3621,-6.4738,0.64,50.40,46750,0.040690,-0.009355,0.0,-25.04355
9,60,41,3.1,0.0,3.9,64.3229,-8.8022,0.64,50.43,46227,0.039716,-0.009067,0.0,-25.04355
